In [2]:
import pandas as pd 
from transformers import T5Tokenizer, Trainer , TrainingArguments, T5ForConditionalGeneration

In [3]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [4]:
train_data.head() 

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [5]:
 train_data["dialogue"][0] 

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [6]:
train_data.shape


(14732, 3)

In [7]:
val_data.shape

(818, 3)

In [8]:
# random sampling 
train_data = train_data.sample(n = 4000, random_state = 42).reset_index (drop= True)
val_data = val_data.sample(n= 500, random_state = 42).reset_index(drop= True)

In [9]:
train_data.shape

(4000, 3)

# Data pre-processing 


In [10]:
import re 

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # lines 
    text = re.sub(r"\s+"," ", text) # spaces 
    text = re.sub(r"<.*?>"," ", text) #html tags <p> <h1>
    text = text.strip().lower()
    return text 

In [11]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"]= train_data["summary"].apply(clean_data) 

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"]= val_data["summary"].apply(clean_data) 



In [12]:
 train_data["dialogue"][0] 

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

# Tokenize 

In [13]:
tokenizer = T5Tokenizer.from_pretrained("t5-small") 
#Load the tokenizer that belongs to the pretrained t5-small model.


In [14]:
# raw data => tokenize inputs for fine-tuning 

def tokenize(data):
    
    # max_length means maximum length of tokens in i/p
    # padding means adding more position if there are less tokens
    # truncation means removing extra tokens than max_length
    inputs = tokenizer(data["dialogue"] , padding = "max_length", max_length = 512, truncation = True)
    targets = tokenizer(data["summary"] , padding = "max_length", max_length = 150, truncation = True)
    # tokenizer automatically returns => input_ids and attention_mask
    

    # combining targets with inputs 
    inputs["labels"] = targets["input_ids"] # so target's input_ids are copied to new col labels of inputs 
    return inputs

In [15]:
train_dataset = train_data.apply(tokenize, axis = 1).tolist() # axis =1 to apply row by row 
val_dataset = val_data.apply(tokenize, axis = 1).tolist() 

In [16]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [17]:
# now we got :
# input ids - dialogue token ids

# 1 => End of sequence
# 0 => paddings

# attention mask  - 1 => valid token id , 0 => invalid token id / padding 
# labels - target => summary token ids

In [18]:
print(len(train_dataset[0]["input_ids"]))
print(len(train_dataset[0]["labels"]))

512
150


# Working with our model

### load the model 

In [19]:

model = T5ForConditionalGeneration.from_pretrained("t5-small")
# its a T5 model class which generates text on the basis of i/p

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

### Fine tuning

In [20]:
import torch 

if torch.backends.mps.is_available() :
    device = torch.device("mps")
elif torch.cuda.is_available() :
    device = torch.device("cuda")
else :
    device = torch.device("cpu")

print("device: ", device)
model.to(device)

device:  mps


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [21]:
# Training arguments => defining settings/configurations for training the model 
 
training_args = TrainingArguments(
    output_dir = "./results", # where we want to save the outputs or checkpoints during training

    num_train_epochs = 6, 
    weight_decay = 0.01,  # reguralization to control overfitting , so that model don't excessively memorize the training data

    per_device_train_batch_size = 8, 
    per_device_eval_batch_size = 8 , 

    eval_strategy = "epoch", 
    save_strategy = "epoch", # after every epoch model will be evaluated and saved 

    warmup_steps = 500 # Initial training steps during which the learning rate gradually increases from very small value to the target learning rate 
)

In [22]:
# creating trainer => it connects the model, training settings, and datasets together.

trainer = Trainer(
    model = model ,
    args= training_args, 
    train_dataset = train_dataset,
    eval_dataset = val_dataset 
)
    

In [23]:
# Train the model 
# trainer.train()

In [24]:
### Save the model 
# model.save_pretrained("./saved_summary_model")
# tokenizer.save_pretrained("./saved_summary_model")

In [26]:
# load the model 

model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

# Test the core logic for summarization

In [31]:
def summarize_dialogue(dialogue) :
    #Inference preprocessing ideally training preprocessing ke compatible hona chahiye.
    dialogue = clean_data(dialogue) # clean 

    # tokenize because T5 directly string process nahi krta 
    inputs = tokenizer(
        dialogue, 
        padding = "max_length" , 
        max_length = 512, 
        truncation = True, 
        return_tensors = "pt" # so it returns pytorch tensors
    ).to(device)

    # generate the summary => token ids will be generated
    model.to(device) # move t5 model 
    targets = model.generate(  # actually generate the summary
        input_ids = inputs["input_ids"], 
        attention_mask = inputs["attention_mask"],
        max_length = 150, #Generated output can contain at most 150 tokens.
        num_beams = 4, # While generating the summary, T5 will keep 4 promising output sequences (paths) at each stage instead of following only one.
        early_stopping = True #If the beam search has finished generating valid sequences earlier, generation can stop rather than unnecessarily continuing toward the maximum.
    )

    # token ids convert to summary => decoding 
    summary = tokenizer.decode(targets[0], skip_special_tokens = True) # target[0]=> because we want token sequence for the first dialogue in the batch.
    return summary 

In [32]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue) 
print(summary)

ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.
